In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

# 1. Cấu hình mô hình và tokenizer
model_id = "Qwen/Qwen2.5-7B-Instruct"  # Hoặc "Qwen/Qwen-7B-Instruct" tùy phiên bản bạn dùng

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = (
    "right"  # Quan trọng đối với việc training các mô hình causal LM
)

# Cấu hình lượng tử hóa 4-bit (QLoRA) để tiết kiệm VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Tải mô hình cơ sở
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Chuẩn bị mô hình cho k-bit training
model = prepare_model_for_kbit_training(model)

# 2. Cấu hình LoRA (PEFT)
peft_config = LoraConfig(
    r=16,  # Rank của ma trận LoRA
    lora_alpha=32,  # Scaling parameter
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()  # Kiểm tra số lượng tham số được train

# 3. Chuẩn bị tập dữ liệu (Dataset mẫu theo định dạng ChatML của Qwen)
# Bạn hãy thay thế bằng tập dữ liệu thực tế của bạn
raw_data = [
    {
        "messages": [
            {
                "role": "system",
                "content": "Bạn là một trợ lý AI hữu ích.",
            },
            {"role": "user", "content": "Thủ đô của Việt Nam là gì?"},
            {
                "role": "assistant",
                "content": "Thủ đô của Việt Nam là Hà Nội.",
            },
        ]
    }
]


# Hàm format dữ liệu sang dạng text chuẩn sử dụng chat template của Qwen
def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example["messages"])):
        text = tokenizer.apply_chat_template(
            example["messages"][i], tokenize=False, add_generation_prompt=False
        )
        output_texts.append(text)
    return {"text": output_texts}


dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_prompts_func, batched=True)

# 4. Cấu hình Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen-lora-output",
    num_train_epochs=3,  # Số lượng epoch huấn luyện
    per_device_train_batch_size=2,  # Giảm xuống 1 nếu thiếu VRAM
    gradient_accumulation_steps=4,  # Tăng lên nếu giảm batch size
    learning_rate=2e-4,  # Learning rate cho LoRA
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",  # Có thể đổi thành "wandb" nếu muốn theo dõi
)

# 5. Khởi tạo SFTTrainer và tiến hành Train
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()

# 6. Lưu LoRA adapter đã train
trainer.model.save_pretrained("./qwen-lora-adapter")
tokenizer.save_pretrained("./qwen-lora-adapter")